In [0]:
 from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/devvrataman@gmail.com/chutiya/1_setup/utilities

In [0]:
print(bronze_schema)

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source', 'orders','Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')
base_path = f's3://sportsbar-devvrat/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'
print('Base Path:', base_path)
print('Landing Path:', landing_path)
print('Processed Path:', processed_path)

# define the tables
bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.sb_fact_{data_source}'


In [0]:
df = spark.read.options(header='True', inferSchema=True).csv(f'{processed_path}/*.csv').withColumn('read_timestamp',F.
current_timestamp()).select('*','_metadata.file_name','_metadata.file_size')

print('Total Rows:', df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
# df.write\
#     .format('delta')\
#     .option('delta.enableChangeDataFeed','true')\
#     .mode('append')\
#     .saveAsTable(bronze_table)

if not spark.catalog.tableExists(bronze_table):
    df.write \
        .format('delta') \
        .option('delta.enableChangeDataFeed', 'true') \
        .mode('overwrite') \
        .saveAsTable(bronze_table)
else:
    bronze_delta = DeltaTable.forName(spark, bronze_table)
    bronze_delta.alias('target').merge(
        df.alias('source'),
        '''target.order_id     = source.order_id
       AND target.product_id   = source.product_id
       AND target.customer_id  = source.customer_id
       AND target.file_name    = source.file_name'''
    ).whenNotMatchedInsertAll().execute()

In [0]:
bronze_table

In [0]:
files = dbutils.fs.ls(landing_path)
files

In [0]:
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f'{processed_path}/{file_info.name}',
        True
    )
    

silver Processing 

In [0]:
df_orders = spark.sql(f'select * from {bronze_table}')


In [0]:
# 1. Keep oly rows where order_qty is present
df_orders = df_orders.filter(F.col('order_qty').isNotNull())

# 2)) Clean customer_id-> keep numeric, or set to 999999
df_orders = df_orders.withColumn(
    'customer_id', F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id')).otherwise(999999)
    .cast('string')
)
# Tuesday, July 01, 2025 -> July 01,2025
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*', '')

)

# 4. Parase order placement date using multiple possible formats
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.coalesce(
        F.try_to_date('order_placement_date', 'MMMM dd, yyyy'),
        F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
        F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
        F.try_to_date('order_placement_date', 'yyyy/MM/dd'),

    )
)

# 5. Drop duplicates

df_orders = df_orders.dropDuplicates(['order_id','customer_id','order_placement_date','order_qty'])

# 6. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))



In [0]:
df_orders.show(5)

In [0]:
print('Number of rows:', df_orders.count())
print('Number of columns:', len(df_orders.columns))

In [0]:
# min and max date 
df_orders.agg(
    F.min('order_placement_date').alias('min_dateu'),
    F.max('order_placement_date').alias('max_date')
).show()

In [0]:
display(df_orders.limit(10))

In [0]:
df_products = spark.table('fmcg.silver.products')
df_products.show(5)
df_joined = df_orders.join(df_products, on='product_id', how='inner').select(df_orders['*'], df_products['product_code'])
df_joined.show(10)

In [0]:
silver_table

In [0]:
# deduplicate on full silver merge key
df_joined = df_joined.dropDuplicates(['order_placement_date', 'order_id', 'product_code', 'customer_id'])
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
silver_table

In [0]:
total_count = spark.sql(f"SELECT COUNT(*) AS total_count FROM {silver_table}")
display(total_count)

### Gold schema

In [0]:
# df_child = spark.sql(f"SELECT date, customer_code, order_qty FROM {gold_table}")
# df_child.show(10)

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

In [0]:
df_gold = df_gold.dropDuplicates(['date', 'order_id', 'product_code', 'customer_code'])
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Merge with parent company

In [0]:
gold_table

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)


In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()